In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI


# This function will load all the variable from .env file and will
# will make available in os.eviron dictionary (env variables)
load_dotenv()

if os.environ.get("GEMINI_API_KEY"):
    print("API key variable exist")
else:
    print("GEMINI_API_KEY not found")

llm_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    google_api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

API key variable exist


In [2]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summery_flag: Literal["positive", "negative"]

llm_structured_output = llm_gemini.with_structured_output(llm_schema)

# **CHAIN WITH Conditional Chains**

In [3]:
#Task 1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please chategorise the movie review as positive or negative : {input}")
])

In [4]:
# Task 2 [LLM]

llm_structured_output = llm_gemini.with_structured_output(llm_schema)

In [5]:

#Task 3 [Custom runnable]

def pydantic_json(input: llm_schema)-> str:

    return input.model_dump()["movie_summery_flag"]

pydantic_json_lambda = RunnableLambda(pydantic_json)

# **Conditional Chain 1**

In [6]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

# TASK - 2 [LLM]

llm_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    google_api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
)

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_gemini | str_parser

# **Conditional Chain 2**

In [7]:
from langchain_core.runnables import RunnableLambda, RunnableBranch

def insta_chain(text:dict):

    text = text["text"]

    #Task -1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a LinkedIn post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    # TASK - 2 [LLM]
    llm_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    google_api_key = os.getenv("GEMINI_API_KEY"),
    temperature = 0
    )

     # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_gemini | str_parser

    result = chain_insta.invoke(text)

    return result


insta_chain_runnable = RunnableLambda(insta_chain)
    

# **Final orchestration**

In [ ]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in str(x).lower(), chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [11]:
final_orchestrator.invoke({"input": "This move was very bad."})

TypeError: string indices must be integers, not 'str'